In [1]:
import sys; sys.path.append('..')
import glob, os
import pandas as pd
from src import settings, TextRetrievalEngine, VisionRetrievalEngine, RAGEngine, EvaluationRunner


c:\Users\kyle0\miniconda3\envs\rag_system\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_files = glob.glob(os.path.join(settings.RAW_DATA_DIR, "*.pdf"))
text_engine = TextRetrievalEngine()
base_retriever, all_docs = text_engine.build_or_load(pdf_files)

vision_retriever = VisionRetrievalEngine().load_index()
engine = RAGEngine(base_retriever, all_docs, vision_retriever)


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\retrieval\text_engine.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33058.72it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ 하드디스크에 저장된 DB와 문서를 불러옵니다!


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\retrieval\text_engine.py:35: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=db_dir, embedding_function=self.embedding_model)


Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 530.96it/s]
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


You are using in-memory collection. This means every image is stored in memory.
You might want to rethink this if you have a large collection!
Loaded 4654 images from 10 JSON files.
✅ 비전 인덱스 'multi_doc_vision_index' 로드 완료


In [3]:
import nltk
import ssl

# Mac/Windows 환경에서 SSL 인증서 에러로 다운로드가 조용히 막히는 것을 방지
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# 필수 토크나이저 데이터 명시적 다운로드 (최신 버전 호환을 위해 punkt_tab 추가)
nltk.download('punkt')
nltk.download('punkt_tab')

print("✅ NLTK 토크나이저 데이터 다운로드 완료! 이제 평가 셀을 다시 실행해 보세요.")

✅ NLTK 토크나이저 데이터 다운로드 완료! 이제 평가 셀을 다시 실행해 보세요.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kyle0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kyle0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# 🎯 5.4. 정성적 평가 및 5.2 성능 측정을 위한 Ground Truth(GT) 세트
eval_dataset = [
    {
        "query": "삼성SDI의 연결 현금흐름표에 기재된 2025년도의 당기순손익은 얼마인가요?",
        "gt_number": "-584,875", 
        "gt_text": "[파일명: [삼성SDI]사업보고서_2026.pdf, 연결 현금흐름표]에 따르면 2025년 당기순손익은 -584,875백만원입니다.", 
        "type": "표 단절 및 동명 기업 혼동"
    },
    {
        "query": "고려제약의 2025년 연구개발비용 총액은 얼마입니까?",
        "gt_number": "2,258,424",
        "gt_text": "[파일명: [고려제약]사업보고서_2026.pdf, 연구개발활동]에 따르면 2025년 연구개발비용 총액은 2,258,424천원입니다.",
        "type": "일반 재무 검색"
    },
    {
        "query": "삼성전자의 2024년 연결재무제표 기준 당기순이익은 얼마인가요?",
        "gt_number": "15,487,100", 
        "gt_text": "[파일명: [삼성전자]사업보고서_2026.pdf, 연결재무제표]에 따르면 2024년 당기순이익은 15,487,100백만원입니다.",
        "type": "일반 표 검색"
    },
    {
        "query": "LG에너지솔루션의 2025년 재무제표 주석에 기재된 '비용의 성격별 분류' 중 '종업원급여'는 얼마인가요?",
        "gt_number": "2,725,702", 
        "gt_text": "[파일명: [LG에너지솔루션]사업보고서_2026.pdf, 주석 (비용의 성격별 분류)]에 따르면 2025년 종업원급여는 2,725,702백만원입니다.",
        "type": "주석 내 특정 표 수치 검색"
    },
    {
        "query": "POSCO홀딩스의 2026년 이사회 의장 이름은 누구인가요?",
        "gt_number": "권태균",
        "gt_text": "[파일명: [POSCO홀딩스]사업보고서_2026.pdf, 이사회 등 회사의 기관에 관한 사항]에 따르면 2026년 이사회 의장은 권태균입니다.",
        "type": "비재무 및 인물 정보 검색"
    },
    {
        "query": "현대해상의 2025년 요약연결재무정보 상 영업수익은 얼마인가요?",
        "gt_number": "17,890,778,942", 
        "gt_text": "[파일명: [현대해상]사업보고서_2026.pdf, 연결포괄손익계산서]에 따르면 2025년 영업수익은 17,890,778,942천원입니다.",
        "type": "보험업 특화 항목 검색"
    },
    {
        "query": "한화손해보험의 요약재무정보에서 자본총계는 얼마로 기록되어 있나요?",
        "gt_number": "2,720,126", 
        "gt_text": "[파일명: [한화손해보험]사업보고서_2026.pdf, 요약재무정보]에 따르면 자본총계는 2,720,126백만원입니다.",
        "type": "보험업 요약재무정보 검색"
    },
    {
        "query": "한미반도체의 2025년 연구개발비 / 매출액 비율은 몇 퍼센트인가요?",
        "gt_number": "4.3", 
        "gt_text": "[파일명: [한미반도체]사업보고서_2026.pdf, 연구개발비용]에 따르면 연구개발비 / 매출액 비율은 4.3%입니다.",
        "type": "표 내 수치 비교 및 분석"
    },
    {
        "query": "한화오션의 2025년 유형자산 장부금액을 연결재무정보에서 찾아주세요.",
        "gt_number": "5,273,173", 
        "gt_text": "[파일명: [한화오션]사업보고서_2026.pdf, 주석]에 따르면 2025년 유형자산 장부금액은 5,273,173백만원입니다.",
        "type": "주석 내 표 데이터 검색"
    },
    {
        "query": "카카오페이의 연결 현금흐름표에서 재무활동현금흐름은 얼마입니까?",
        "gt_number": "2,671,245,155", 
        "gt_text": "[파일명: [카카오페이]사업보고서_2026.pdf, 연결 현금흐름표]에 따르면 재무활동현금흐름은 2,671,245,155원입니다.",
        "type": "현금흐름표 긴 표 단절 테스트"
    }
]

In [5]:
runner = EvaluationRunner(engine)
df_results = runner.run_full_evaluation(eval_dataset)

summary = df_results.groupby("Model")[["Exact_Match", "ROUGE-L", "BLEU", "LLM_Judge"]].mean()
display(summary.reindex([
    "Method 0 (Baseline)", "Method 1 (Vision Real)", "Method 2 (Dual Basic)",
    "Method 3 (No-Filter)", "Method 3 (No-Window)", "Method 3 (SOTA)"
]))


Loading weights: 100%|██████████| 149/149 [00:00<00:00, 29766.67it/s]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



▶️ [평가 1/10] 삼성SDI의 연결 현금흐름표에 기재된 2025년도의 당기순손익은 얼마인가요?


It is a prefill stage but The `token_type_ids` is not provided. We recommend passing `token_type_ids` to the model to prevent bad attention masking.


🔄 [Agent Loop 1] 분석 중인 페이지: [132, 133, 134]
🔄 [Agent Loop 1] 분석 중인 페이지: [236, 237, 238]
🔄 [Agent Loop 1] 분석 중인 페이지: [235, 236, 237]
🔄 [Agent Loop 1] 분석 중인 페이지: [235, 236, 237]

▶️ [평가 2/10] 고려제약의 2025년 연구개발비용 총액은 얼마입니까?
🔄 [Agent Loop 1] 분석 중인 페이지: [31, 32, 33]
🔄 [Agent Loop 1] 분석 중인 페이지: [31, 32, 33]
🔄 [Agent Loop 1] 분석 중인 페이지: [56, 57, 58]
🔄 [Agent Loop 1] 분석 중인 페이지: [29, 30, 31]
🔄 [Agent Loop 1] 분석 중인 페이지: [442, 443, 444]
🔄 [Agent Loop 1] 분석 중인 페이지: [49, 50, 51]
🔄 [Agent Loop 1] 분석 중인 페이지: [49, 50, 51]
🔄 [Agent Loop 1] 분석 중인 페이지: [49, 50, 51]

▶️ [평가 3/10] 삼성전자의 2024년 연결재무제표 기준 당기순이익은 얼마인가요?
🔄 [Agent Loop 1] 분석 중인 페이지: [327, 328, 329]
🔄 [Agent Loop 1] 분석 중인 페이지: [556, 557, 558]
🔄 [Agent Loop 1] 분석 중인 페이지: [558, 559, 560]
🔄 [Agent Loop 1] 분석 중인 페이지: [30, 31, 32]
🔄 [Agent Loop 1] 분석 중인 페이지: [391, 392, 393]
🔄 [Agent Loop 1] 분석 중인 페이지: [409, 410, 411]
🔄 [Agent Loop 1] 분석 중인 페이지: [27, 28, 29]
🔄 [Agent Loop 1] 분석 중인 페이지: [660, 661, 662]
🔄 [Agent Loop 1] 분석 중인 페이지: [555, 556, 557]
🔄 [Agent 

,Exact_Match,ROUGE-L,BLEU,LLM_Judge
Model,,,,
Method 0 (Baseline),0.4,0.3520,0.0371,3.35
Method 1 (Vision Real),0.0,0.2583,0.0082,2.05
Method 2 (Dual Basic),0.1,0.2460,0.0308,2.15
Method 3 (No-Filter),0.3,0.2078,0.0846,2.00
Method 3 (No-Window),0.4,0.3008,0.1052,2.70
Method 3 (SOTA),0.4,0.3008,0.1060,2.70
